# 033 — Signal Evaluation (model-independent reference)

Ranks every hidden-detail signal the pipeline produces, on **real artworks**, without a hand-drawn mask.

`031`/`032` compare models on `mae`/`ssim`/`psnr`, which measure how well `mu` reproduces the IR — not whether the residual reveals anything. The only annotated reference available (`modern`) is a single purpose-built pair: no statistical power, and materially unlike an aged painting. This notebook uses two references that need no annotation and apply to the whole test fold:

- **Detection** (`scripts.pseudo_mask` + `scripts.detection`) — a pseudo ground truth computed from the data alone: *structure present in the IR and not explained by the RGB*. Every candidate signal is scored against it with AUROC / average precision.
- **Stroke coherence** (`scripts.stroke_stats`) — fully unsupervised. An underdrawing is made of oriented, elongated strokes; prediction noise is isotropic. Structure-tensor coherence separates the two without knowing where the strokes are.

Neither measures "found the underdrawing" — they measure *necessary* properties of a signal that would. They fail in different ways, so **agreement between the two is worth more than either alone**, which is what §6 checks.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from scipy.stats import spearmanr

from scripts.calibration import learned_zscore
from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    grouped_train_val_test_split,
    load_image_pairs,
)
from scripts.delta_analysis import compute_local_stats, compute_ssim_components
from scripts.detection import rank_signals
from scripts.pseudo_mask import cross_modal_pseudo_mask
from scripts.stroke_stats import stroke_coherence
from scripts.trainer import load_model
from scripts.trainer_nll import load_model_nll
from scripts.visualization_nll import plot_signal_comparison

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Test set

Same grouped-by-artwork split as `030`/`032`, so the images scored here are the ones no checkpoint was trained or validated on.

In [ ]:
N_EVAL = 6  # test images to evaluate over
MASK_PERCENTILE = 95.0  # top 5% of the pseudo-mask score counts as positive

pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
_, _, test_pairs = grouped_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    seed=settings.SEED,
)
eval_pairs = test_pairs[:N_EVAL]
print(f"Test pairs: {len(test_pairs)} | evaluating on: {len(eval_pairs)}")

## 2. What the pseudo-mask sees

Two factors multiplied: is there IR structure here at all, and is it explained by the RGB? The **absolute** correlation is used — an inverted IR (dark paint reading bright) is fully explained, just with opposite sign.

In [ ]:
rgb_batch, ir_batch = next(
    iter(build_dataset(eval_pairs[:1], batch_size=1, augment=False, shuffle=False))
)
rgb_demo = rgb_batch[0].numpy()
ir_demo = ir_batch[0].numpy().squeeze()

mask_demo = cross_modal_pseudo_mask(rgb_demo, ir_demo)

fig = plot_signal_comparison(
    ir_demo,
    {
        "IR contrast": mask_demo.ir_contrast,
        "1 - |corr(IR, RGB)|": 1.0 - mask_demo.cross_structure,
        "pseudo-mask score": mask_demo.score,
        f"binary mask (p{MASK_PERCENTILE:g})": mask_demo.binarize(
            MASK_PERCENTILE
        ).astype(float),
    },
    title=f"Cross-modal pseudo ground truth — {Path(eval_pairs[0][0]).stem}",
    vrange=(0.0, 1.0),
)
plt.show()

## 3. Candidate signals, per model

Every signal is a magnitude map (higher = more likely hidden detail), so they are all scored on identical terms:

- deterministic architectures → raw delta `|IR - pred|` and structural delta `1 - structure`;
- NLL architectures, both loss variants → raw delta `|IR - mu|` and `|z| = |(IR - mu) / sigma|`.

In [ ]:
# Every architecture with a checkpoint on disk is scored; missing ones are
# reported and skipped, so this runs against whatever has been trained so far.
DET_ARCHS = [
    "unet",
    "resunet",
    "attention_unet",
    "efficientnet_unet",
    "unet_v2",
    "unet_restormer",
]
NLL_ARCHS = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
BETA_MODEL_DIR = settings.MODELS_DIR / "nll_beta"
BETA = 0.5

det_models = {}
for arch in DET_ARCHS:
    try:
        det_models[arch] = load_model(arch, model_dir=settings.MODELS_DIR)
    except (FileNotFoundError, OSError) as exc:
        print(f"[skip] {exc}")

nll_models = {}
for arch in NLL_ARCHS:
    for variant, model_dir in (
        ("gaussian", settings.MODELS_DIR),
        ("beta", BETA_MODEL_DIR),
    ):
        loss_name = "gaussian_nll" if variant == "gaussian" else "beta_nll"
        try:
            nll_models[f"{arch} ({variant})"] = load_model_nll(
                arch, model_dir=model_dir, loss_name=loss_name, beta=BETA
            )
        except (FileNotFoundError, OSError) as exc:
            print(f"[skip] {exc}")

n_signals = 2 * (len(det_models) + len(nll_models))
print(f"\nLoaded {len(det_models)} deterministic, {len(nll_models)} NLL models")
print(f"-> {n_signals} candidate signals per image")

# Fail here rather than four cells downstream: below three signals the ranking
# has nothing to rank and §6's rank correlation is trivially +/-1, which reads
# like a result but is an artefact of the sample size.
if n_signals < 3:
    raise RuntimeError(
        f"Only {n_signals} candidate signals available — at least 3 are needed "
        "for the ranking and the rank correlation to mean anything. Train the "
        "missing architectures (020/021/022) or point MODELS_DIR at a tree that "
        "holds their checkpoints."
    )

In [ ]:
def structural_delta(ir, pred):
    """``1 - local SSIM structure`` between the real and predicted IR.

    Computed directly rather than through ``analyze_delta``, which would also
    build the per-zone Wasserstein map — a Python loop over every 32x32 tile —
    only for this one field to be read off the result.
    """
    stats = compute_local_stats(ir, pred)
    return 1.0 - compute_ssim_components(stats).structure


def signals_for(rgb_batch, ir):
    """Build every candidate magnitude map for one image."""
    signals = {}
    for name, model in det_models.items():
        pred = model.predict(rgb_batch, verbose=0)[0, ..., 0]
        signals[f"{name} delta"] = np.abs(ir - pred)
        signals[f"{name} structural"] = structural_delta(ir, pred)
    for name, model in nll_models.items():
        pred = model.predict(rgb_batch, verbose=0)[0]
        mu, sigma = pred[..., 0], np.exp(0.5 * pred[..., 1])
        signals[f"{name} delta"] = np.abs(ir - mu)
        signals[f"{name} |z|"] = np.abs(learned_zscore(ir, mu, sigma))
    return signals


auroc_rows: dict[str, list[float]] = {}
ap_rows: dict[str, list[float]] = {}
lift_rows: dict[str, list[float]] = {}
coherence_rows: dict[str, list[float]] = {}

for (rgb_batch, ir_batch), pair in zip(
    build_dataset(eval_pairs, batch_size=1, augment=False, shuffle=False), eval_pairs
):
    ir = ir_batch[0].numpy().squeeze()
    mask = cross_modal_pseudo_mask(rgb_batch[0].numpy(), ir).binarize(MASK_PERCENTILE)
    signals = signals_for(rgb_batch, ir)

    for name, result in rank_signals(signals, mask).items():
        auroc_rows.setdefault(name, []).append(result.auroc)
        ap_rows.setdefault(name, []).append(result.average_precision)
        lift_rows.setdefault(name, []).append(result.lift)
    for name, signal in signals.items():
        coherence_rows.setdefault(name, []).append(stroke_coherence(signal).coherence)

    print(f"{Path(pair[0]).stem}: {len(signals)} signals scored")

## 4. Detection ranking

`auroc` is prevalence-independent and is the primary ranking. `lift` is average precision relative to chance: how much better than random the top of the ranking is. `std` across images is the honest error bar — a difference smaller than it means nothing.

**Read like with like.** The pseudo-mask shares its mathematical form with the structural delta — both are built on the same local windowed structure statistic — so a structural signal scores well against it *by construction*, not on merit. Comparing architectures **within** one signal type is sound; comparing signal types against each other on this axis alone is not, which is what §5 is for.


In [ ]:
def summarise(rows):
    return {name: (float(np.mean(v)), float(np.std(v))) for name, v in rows.items()}


auroc_mean = summarise(auroc_rows)
ap_mean = summarise(ap_rows)
lift_mean = summarise(lift_rows)
ranking = sorted(auroc_mean, key=lambda n: auroc_mean[n][0], reverse=True)

col_w = 30
print("signal".ljust(col_w) + "auroc".ljust(18) + "avg_prec".ljust(18) + "lift")
print("-" * (col_w + 54))
for name in ranking:
    a, a_sd = auroc_mean[name]
    p, _ = ap_mean[name]
    fold, _ = lift_mean[name]
    print(f"{name.ljust(col_w)}{a:.4f} ± {a_sd:.3f}   {p:.4f}            {fold:.2f}x")

In [ ]:
top = ranking[:12]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(top) + 2))
y = np.arange(len(top))
ax.barh(
    y,
    [auroc_mean[n][0] for n in top],
    xerr=[auroc_mean[n][1] for n in top],
    capsize=3,
)
ax.axvline(0.5, color="k", ls="--", lw=1, label="chance")
ax.set_yticks(y)
ax.set_yticklabels(top, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("AUROC vs. cross-modal pseudo-mask")
ax.set_title(f"Detection ranking — mean ± std over {N_EVAL} test images")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Stroke coherence — the unsupervised axis

No reference at all: how much of each signal is oriented, line-like structure rather than isotropic noise. Read it with suspicion — a model with patch-stitching seams or `Conv2DTranspose` checkerboarding scores high while revealing nothing.

In [ ]:
coherence_mean = summarise(coherence_rows)
by_coherence = sorted(coherence_mean, key=lambda n: coherence_mean[n][0], reverse=True)

print("signal".ljust(col_w) + "coherence")
print("-" * (col_w + 20))
for name in by_coherence:
    c, c_sd = coherence_mean[name]
    print(f"{name.ljust(col_w)}{c:.4f} ± {c_sd:.3f}")

## 6. Do the two axes agree?

They are wrong in different ways, so their agreement is the real result. A high rank correlation means both references point at the same signal and the conclusion is robust; a low one means at least one of them is measuring an artefact, and neither ranking should be trusted on its own.

In [ ]:
names = list(auroc_mean)
x = [auroc_mean[n][0] for n in names]
y = [coherence_mean[n][0] for n in names]
rho = spearmanr(x, y).statistic

print(f"Spearman(auroc, coherence) = {rho:.3f} over {len(names)} signals")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(x, y)
for name, xi, yi in zip(names, x, y):
    ax.annotate(name, (xi, yi), fontsize=6, alpha=0.7)
ax.axvline(0.5, color="k", ls="--", lw=1)
ax.set_xlabel("AUROC vs. pseudo-mask (supervised axis)")
ax.set_ylabel("stroke coherence (unsupervised axis)")
ax.set_title(f"Agreement between the two references — rho = {rho:.3f}")
plt.tight_layout()
plt.show()